In [1]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import pandas as pd


In [2]:
import numpy as np
from scipy.optimize import minimize

EPS = 1e-12

# ============================================================
# 1) Downstream competition
#    - n=1: monopoly full coverage, p = v - lambda
#    - n=2: Hotelling line (your closed form)
#    - n=3: Salop circle (standard)
# ============================================================

def solve_hotelling2(phi, lambd, idx):
    i, j = idx
    phi = np.asarray(phi, dtype=float)
    p_i = (2 * phi[i] + phi[j]) / 3 + lambd
    p_j = (2 * phi[j] + phi[i]) / 3 + lambd
    prices = np.zeros(len(phi))
    prices[i] = p_i
    prices[j] = p_j
    return prices

def shares_hotelling2(prices, lambd, idx):
    i, j = idx
    p_i, p_j = prices[i], prices[j]
    s_i = 0.5 + (p_j - p_i) / (2 * lambd)
    s_j = 1.0 - s_i
    s = np.zeros(len(prices))
    s[i], s[j] = s_i, s_j
    return s

def solve_hotelling3(phi, lambd):
    """
    Standard 3-firm Salop circle price equilibrium:
      (4I - shift - shift^T) p = 2*phi + (2*lambd/3)*1
    """
    phi = np.asarray(phi, dtype=float)
    n = 3
    assert len(phi) == 3

    A = 4*np.eye(n)
    for i in range(n):
        A[i, (i-1) % n] -= 1.0
        A[i, (i+1) % n] -= 1.0

    b = 2*phi + (2*lambd/n)*np.ones(n)
    return np.linalg.solve(A, b)

def shares_hotelling3(prices, lambd):
    """
    Standard 3-firm Salop circle shares:
      s_i = 1/3 + (p_{i-1}+p_{i+1}-2 p_i)/(2*lambd)
    """
    p = np.asarray(prices, dtype=float)
    n = 3
    assert len(p) == 3

    s = np.zeros(n)
    for i in range(n):
        s[i] = 1/n + (p[(i-1) % n] + p[(i+1) % n] - 2*p[i])/(2*lambd)

    # numerical guard
    s = np.maximum(s, 0.0)
    tot = s.sum()
    if tot > 0:
        s /= tot
    return s

def calc_price(phi, lambd, v):
    """
    v = willingness to pay (your WTP)
    lambd = transport parameter (your COST)
    Full coverage assumption.
    """
    phi = np.asarray(phi, dtype=float)
    n = len(phi)

    if n == 1:
        # monopoly premium (full coverage Hotelling-style)
        return np.array([max(0.0, v - lambd)], dtype=float)

    elif n == 2:
        return solve_hotelling2(phi, lambd, [0, 1])

    elif n == 3:
        return solve_hotelling3(phi, lambd)

    else:
        raise NotImplementedError("Only supports n=1,2,3.")

def calc_s(phi, lambd, v):
    phi = np.asarray(phi, dtype=float)
    n = len(phi)
    prices = calc_price(phi, lambd, v)

    if n == 1:
        return np.array([1.0], dtype=float)

    elif n == 2:
        return shares_hotelling2(prices, lambd, [0, 1])

    elif n == 3:
        return shares_hotelling3(prices, lambd)

    else:
        raise NotImplementedError("Only supports n=1,2,3.")

def calc_profits(phi, lambd, v, mc):
    phi = np.asarray(phi, dtype=float)
    mc = np.asarray(mc, dtype=float)

    s = calc_s(phi, lambd, v)
    prices = calc_price(phi, lambd, v)

    hospital_profit = np.sum(s * (phi - mc))
    insurer_profits = s * (prices - phi)
    return hospital_profit, insurer_profits, s, prices


# ============================================================
# 2) Analytic starting guesses (so your notebook works)
#    NOTE: n=3 formulas below are for the *Salop* version now.
# ============================================================

def analytic_phi(lambd, v, n, belief="active", beta=0.5):
    """
    Returns an analytic starting value (and for n<=2, the true closed form under Hotelling).
    For n=3 Salop, these are useful closed-form benchmarks under the same "local" logic.
    """
    if n == 0:
        return 0.0

    if n == 1:
        # Monopoly bargain with p=v-lambda and Nash weight beta:
        # maximize (p-phi)^beta * phi^(1-beta) => phi = (1-beta)*p
        p = max(0.0, v - lambd)
        return (1-beta)*p

    if n == 2:
        if belief == "active":
            # your paper's Hotelling(2) active beliefs formula at beta=1/2:
            # phi = 3/4*lambda + (v-lambda)/2 = v/2 + lambda/4
            # generalized beta version not needed here; keep beta=1/2 default
            return 0.75*lambd + 0.5*(v - lambd)
        elif belief == "passive":
            return 1.5*lambd

    if n == 3:
        # With standard Salop(3) + fixed-disagreement passive beliefs,
        # the "increment" is not 3/4*lambda. The Salop(3) local increment is 5/36*lambda.
        # Passive beliefs disagreement is (2/3)*phi, so:
        #   phi = 5/36*lambda + (2/3)*phi  => phi = (5/12)*lambda
        if belief == "passive":
            return (5/12)*lambd

        # For active beliefs: use disagreement payoff from 2-insurer Hotelling active,
        # D = v/2 + lambda/4, then add Salop(3) increment 5/36*lambda:
        #   phi = v/2 + lambda/4 + 5/36*lambda = v/2 + 7/18*lambda
        if belief == "active":
            return 0.5*v + (7/18)*lambd

    return 0.5*v


# ============================================================
# 3) Nash objective helper
# ============================================================

def nash_obj_helper(phi_vec, lambd, v, mc, betas, disagreement):
    hospital_profit, insurer_profits, s, prices = calc_profits(phi_vec, lambd, v, mc)

    if hospital_profit <= EPS:
        return 1e9
    if hospital_profit - disagreement <= EPS:
        return 1e9
    if insurer_profits[0] <= EPS:
        return 1e9

    beta = betas[0]
    obj = beta*np.log(insurer_profits[0]) + (1-beta)*np.log(hospital_profit - disagreement)
    return -obj


# ============================================================
# 4) Disagreement payoffs
# ============================================================

def passive_disagreement(base_phi_vec, lambd, v, mc):
    """
    Passive beliefs (no recapture): disagreement payoff is computed at the BASE point
    and held fixed within each bilateral Nash problem.
    """
    s_base = calc_s(base_phi_vec, lambd, v)
    base_phi_vec = np.asarray(base_phi_vec, dtype=float)
    mc = np.asarray(mc, dtype=float)

    return np.sum(s_base*(base_phi_vec-mc)) - s_base[0]*(base_phi_vec[0]-mc[0])


def recursive_disagreement_active(base_phi_vec, lambd, v, mc, betas, analytic_base=True):
    """
    Active beliefs: if insurer 0 drops out, solve the (n-1)-insurer bargaining game (active beliefs)
    and return the hospital's profit in that subgame.
    """
    n = len(base_phi_vec)
    if n <= 1:
        return 0.0

    # Subgame with insurers 1..n-1
    phi_sub_init = np.array([analytic_phi(lambd, v, n-1, belief="active")]*(n-1), dtype=float)
    mc_sub = np.asarray(mc[1:], dtype=float)
    betas_sub = np.asarray(betas[1:], dtype=float)

    phi_sub = solve_nash(phi_sub_init, lambd, v, mc_sub, betas_sub,
                         active=True, analytic_base=analytic_base)

    hosp_profit_sub, _, _, _ = calc_profits(phi_sub, lambd, v, mc_sub)
    return hosp_profit_sub


# ============================================================
# 5) Solver (coordinate descent)
# ============================================================

def solve_nash(phi_init, lambd, v, mc, betas=None,
              active=False, maxiter=100, tol=1e-7,
              analytic_base=True, verbose=False):
    """
    Coordinate descent for n-firm Nash-in-Nash bargaining.
    Keeps your old call signature: solve_nash(phi_init, cost, wtp, mc, betas, active=...).
    """

    phi_init = np.asarray(phi_init, dtype=float)
    mc = np.asarray(mc, dtype=float)
    n = len(phi_init)

    if betas is None:
        betas = np.array([0.5]*n, dtype=float)
    else:
        betas = np.asarray(betas, dtype=float)

    # Optional analytic base cases to stabilize recursion
    if analytic_base and n <= 2:
        phi0 = analytic_phi(lambd, v, n, belief=("active" if active else "passive"))
        return np.full(n, phi0, dtype=float)

    phi = phi_init.copy()
    phi_prev = phi.copy()

    for it in range(maxiter):
        phi_prev[:] = phi[:]

        for k in range(n):
            base_phi = phi_prev.copy()

            # roll so bargainer k becomes index 0
            base_phi_roll = np.roll(base_phi, -k)
            mc_roll = np.roll(mc, -k)
            betas_roll = np.roll(betas, -k)

            # Compute disagreement ONCE at base point, hold fixed in scalar minimization
            if active:
                D_fixed = recursive_disagreement_active(base_phi_roll, lambd, v, mc_roll, betas_roll,
                                                        analytic_base=analytic_base)
            else:
                D_fixed = passive_disagreement(base_phi_roll, lambd, v, mc_roll)

            def helper(phi_k_scalar):
                trial_phi = phi_prev.copy()
                trial_phi[k] = float(phi_k_scalar[0])

                trial_phi_roll = np.roll(trial_phi, -k)
                return nash_obj_helper(trial_phi_roll, lambd, v, mc_roll, betas_roll, D_fixed)

            res = minimize(helper, [phi[k]], method='Nelder-Mead', options={'disp': False})
            phi[k] = res.x[0]

        diff = np.max(np.abs(phi - phi_prev))
        if verbose:
            print(f"Iter {it}, phi={phi}, diff={diff}")
        if diff < tol:
            break

    return phi

In [3]:


phi = np.array([14.4375,13.75,13.75])
lambd = 5.0  # try as needed
wtp = 25.0  # just needs to be larger than all phi+lambd
prices = calc_price(phi, lambd, wtp)
print("Prices:", prices)



Prices: [15.82916667 15.55416667 15.55416667]


In [4]:
calc_price([ 0. ,  18.75 ,18.75], lambd, wtp)

array([ 9.16666667, 16.66666667, 16.66666667])

In [5]:
lambd = 5.0  # try as needed
wtp = 25.0  # just needs to be larger than all phi+lambd
s = calc_s([14.4375,13.75,13.75], lambd, wtp)
print("shares:", s )

shares: [0.27833333 0.36083333 0.36083333]


In [6]:
# --- Model run/prints ---
COST = 6
WTP = 25
MC = np.array([0, 0, 0])
betas = [0.5, 0.5, 0.5]
betas2 = [0.5, 0.5]

n =3
phi_init_pass = [analytic_phi(COST, WTP, n, belief='passive')]*n
phi_passive = solve_nash(phi_init_pass, COST, WTP, np.zeros(n), [0.5]*n, active=False)
print("PASSIVE φ:", phi_passive)
hospital_profit, insurer_profits, s, prices = calc_profits(phi_passive, COST, WTP, np.zeros(n))
print("   hospital profit:", hospital_profit, "insurers:", insurer_profits, "shares:", s, "prices:", prices)
print("Analytic φ (passive):", analytic_phi(COST, WTP, n, "passive"))

PASSIVE φ: [2.5 2.5 2.5]
   hospital profit: 2.5 insurers: [0.66666667 0.66666667 0.66666667] shares: [0.33333333 0.33333333 0.33333333] prices: [4.5 4.5 4.5]
Analytic φ (passive): 2.5


In [7]:
# --- Model run/prints ---
COST = 5
WTP = 26

MC = np.array([0, 0, 0])
betas = [0.5, 0.5, 0.5]
betas2 = [0.5, 0.5]

n =3
phi_init_act = [analytic_phi(COST, WTP, n, belief='active')]*n
phi_active = solve_nash(phi_init_act, COST, WTP, np.zeros(n), [0.5]*n, active=True)
print("ACTIVE φ:", phi_active)
hospital_profit_a, insurer_profits_a, s_a, prices_a = calc_profits(phi_active, COST, WTP, np.zeros(n))
print("   hospital profit:", hospital_profit_a, "insurers:", insurer_profits_a, "shares:", s_a, "prices:", prices_a)
print("Analytic φ (active):", analytic_phi(COST, WTP, n, "active"))

ACTIVE φ: [14.94444444 14.94444444 14.94444444]
   hospital profit: 14.944444444444443 insurers: [0.55555556 0.55555556 0.55555556] shares: [0.33333333 0.33333333 0.33333333] prices: [16.61111111 16.61111111 16.61111111]
Analytic φ (active): 14.944444444444445


In [8]:
def run_active_passive_tables():
    import numpy as np
    import pandas as pd

    COST = 5
    l_fixed = 5
    v_vals = np.arange(24, 28)  # 24 to 28 inclusive
    n = 3
    MC = np.array([0, 0, 0])
    betas = [0.5]*n

    # --- Table 1: Vary v, fix λ=5 ---
    headers = ["WTP (v)", "phi (active)", "hospital_profit_a", "insurer_profits_a", "shares_a", "prices_a"]
    table_active_v = []

    headers2 = ["WTP (v)", "phi (passive)", "hospital_profit", "insurer_profits", "shares", "prices"]
    table_passive_v = []

    for v in v_vals:
        phi_init_act = [analytic_phi(l_fixed, v, n, belief='active')]*n
        phi_active = solve_nash(phi_init_act, l_fixed, v, MC, betas, active=True)
        hospital_profit_a, insurer_profits_a, s_a, prices_a = calc_profits(phi_active, l_fixed, v, MC)
        # Take only first value from lists:
        row = [v, 
               np.round(phi_active[0],2),
               np.round(hospital_profit_a,2),
               np.round(insurer_profits_a[0],2),   # <-- only first item
               np.round(s_a[0],2),                 # <-- only first item
               np.round(prices_a[0],2)]            # <-- only first item
        table_active_v.append(row)

        phi_init_pass = [analytic_phi(l_fixed, v, n, belief='passive')]*n
        phi_passive = solve_nash(phi_init_pass, l_fixed, v, MC, betas, active=False)
        hospital_profit, insurer_profits, s, prices = calc_profits(phi_passive, l_fixed, v, MC)
        row2 = [v,
                np.round(phi_passive[0],2),
                np.round(hospital_profit,2),
                np.round(insurer_profits[0],2),    # <-- only first item
                np.round(s[0],2),                  # <-- only first item
                np.round(prices[0],2)]             # <-- only first item
        table_passive_v.append(row2)

    # --- Table 2: Vary λ, fix v=25 ---
    lambdas = np.arange(4, 8)  # 4 to 8 inclusive

    headers3 = ["COST (λ)", "phi (active)", "hospital_profit_a", "insurer_profits_a", "shares_a", "prices_a"]
    table_active_l = []

    headers4 = ["COST (λ)", "phi (passive)", "hospital_profit", "insurer_profits", "shares", "prices"]
    table_passive_l = []

    for lmbd in lambdas:
        phi_init_act = [analytic_phi(lmbd, 25, n, belief='active')]*n
        phi_active = solve_nash(phi_init_act, lmbd, 25, MC, betas, active=True)
        hospital_profit_a, insurer_profits_a, s_a, prices_a = calc_profits(phi_active, lmbd, 25, MC)
        row = [lmbd,
               np.round(phi_active[0],2),
               np.round(hospital_profit_a,2),
               np.round(insurer_profits_a[0],2),   # <-- only first item
               np.round(s_a[0],2),                 # <-- only first item
               np.round(prices_a[0],2)]            # <-- only first item
        table_active_l.append(row)

        phi_init_pass = [analytic_phi(lmbd, 25, n, belief='passive')]*n
        phi_passive = solve_nash(phi_init_pass, lmbd, 25, MC, betas, active=False)
        hospital_profit, insurer_profits, s, prices = calc_profits(phi_passive, lmbd, 25, MC)
        row2 = [lmbd,
                np.round(phi_passive[0],2),
                np.round(hospital_profit,2),
                np.round(insurer_profits[0],2),    # <-- only first item
                np.round(s[0],2),                  # <-- only first item
                np.round(prices[0],2)]             # <-- only first item
        table_passive_l.append(row2)

    # --- PRINT TABLES ---
    df1 = pd.DataFrame(table_active_v, columns=headers)
    df2 = pd.DataFrame(table_passive_v, columns=headers2)
    df3 = pd.DataFrame(table_active_l, columns=headers3)
    df4 = pd.DataFrame(table_passive_l, columns=headers4)

    with pd.ExcelWriter('hotelling_results.xlsx') as writer:
        df1_sheet = "Active_WTP"
        df2_sheet = "Passive_WTP"
        df3_sheet = "Active_COST"
        df4_sheet = "Passive_COST"

        df1_title = pd.DataFrame([["=== ACTIVE: Varying WTP (v) with λ=5 ==="]])
        df1_title.to_excel(writer, sheet_name=df1_sheet, header=False, index=False, startrow=0)
        df1.to_excel(writer, sheet_name=df1_sheet, index=False, startrow=2)

        df2_title = pd.DataFrame([["=== PASSIVE: Varying WTP (v) with λ=5 ==="]])
        df2_title.to_excel(writer, sheet_name=df2_sheet, header=False, index=False, startrow=0)
        df2.to_excel(writer, sheet_name=df2_sheet, index=False, startrow=2)

        df3_title = pd.DataFrame([["=== ACTIVE: Varying COST (λ) with v=25 ==="]])
        df3_title.to_excel(writer, sheet_name=df3_sheet, header=False, index=False, startrow=0)
        df3.to_excel(writer, sheet_name=df3_sheet, index=False, startrow=2)

        df4_title = pd.DataFrame([["=== PASSIVE: Varying COST (λ) with v=25 ==="]])
        df4_title.to_excel(writer, sheet_name=df4_sheet, header=False, index=False, startrow=0)
        df4.to_excel(writer, sheet_name=df4_sheet, index=False, startrow=2)

    print("Results saved to hotelling_results.xlsx")

    # Only print first row of each DataFrame (as requested)
    print(df1.head(1))
    print(df2.head(1))
    print(df3.head(1))
    print(df4.head(1))

    return df1,df2,df3,df4
# Run
run_active_passive_tables()

Results saved to hotelling_results.xlsx
   WTP (v)  phi (active)  hospital_profit_a  insurer_profits_a  shares_a  \
0       24         13.94              13.94               0.56      0.33   

   prices_a  
0     15.61  
   WTP (v)  phi (passive)  hospital_profit  insurer_profits  shares  prices
0       24           2.08             2.08             0.56    0.33    3.75
   COST (λ)  phi (active)  hospital_profit_a  insurer_profits_a  shares_a  \
0         4         14.06              14.06               0.44      0.33   

   prices_a  
0     15.39  
   COST (λ)  phi (passive)  hospital_profit  insurer_profits  shares  prices
0         4           1.67             1.67             0.44    0.33     3.0


(   WTP (v)  phi (active)  hospital_profit_a  insurer_profits_a  shares_a  \
 0       24         13.94              13.94               0.56      0.33   
 1       25         14.44              14.44               0.56      0.33   
 2       26         14.94              14.94               0.56      0.33   
 3       27         15.44              15.44               0.56      0.33   
 
    prices_a  
 0     15.61  
 1     16.11  
 2     16.61  
 3     17.11  ,
    WTP (v)  phi (passive)  hospital_profit  insurer_profits  shares  prices
 0       24           2.08             2.08             0.56    0.33    3.75
 1       25           2.08             2.08             0.56    0.33    3.75
 2       26           2.08             2.08             0.56    0.33    3.75
 3       27           2.08             2.08             0.56    0.33    3.75,
    COST (λ)  phi (active)  hospital_profit_a  insurer_profits_a  shares_a  \
 0         4         14.06              14.06               0.44      0.3